In [16]:
from datasets import load_dataset

dataset = load_dataset("nntsuzu/KFTT")

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 218038
    })
})


In [17]:
empty_count = sum(
    1 for d in dataset['train']
    if not d['translation']['en'].strip() or not d['translation']['ja'].strip()
)
print(empty_count)

1


In [18]:
def is_valid(example):
    t = example['translation']
    return len(t['en'].strip()) > 0 and len(t['ja'].strip()) > 0

dataset['train'] = dataset['train'].filter(is_valid)

In [19]:
print(dataset['train'][0])

{'translation': {'en': 'Known as Sesshu (1420 - 1506), he was an ink painter and Zen monk active in the Muromachi period in the latter half of the 15th century, and was called a master painter.', 'ja': '雪舟（せっしゅう、1420年（応永27年）-1506年（永正3年））は号で、15世紀後半室町時代に活躍した水墨画家・禅僧で、画聖とも称えられる。'}}


In [20]:
english_lines = []
japanese_lines = []

for data in dataset['train']:
    translation = data['translation']
    english_lines.append(translation['en'])
    japanese_lines.append(translation['ja'])

with open('english.txt', 'w') as f:
    f.write(" ".join(english_lines))

with open('japanese.txt', 'w') as f:
    f.write(" ".join(japanese_lines))

In [21]:
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import WordPieceTrainer

# english
english_tokenizer = Tokenizer(WordPiece(unk_token="[UNK]"))
english_tokenizer.pre_tokenizer = Whitespace()

# japanese
japanese_tokenizer = Tokenizer(WordPiece(unk_token="[UNK]"))
japanese_tokenizer.pre_tokenizer = Whitespace()

# added [SOS]/[EOS] - the decoder needs a start token to kick off generation
# and an end token so it (and inference) knows when to stop
trainer = WordPieceTrainer(
    vocab_size=8000,
    special_tokens=["[UNK]", "[PAD]", "[SOS]", "[EOS]"]
)

english_tokenizer.train(["english.txt"], trainer)
japanese_tokenizer.train(["japanese.txt"], trainer)

ENGLISH_VOCAB_SIZE = english_tokenizer.get_vocab_size()
JAPANESE_VOCAB_SIZE = japanese_tokenizer.get_vocab_size()
print("English vocab size:", ENGLISH_VOCAB_SIZE)
print("Japanese vocab size:", JAPANESE_VOCAB_SIZE)








English vocab size: 8000
Japanese vocab size: 11862


In [22]:
import torch
from torch.utils.data import Dataset as TorchDataset

MAX_LEN = 64

class TranslationDataset(TorchDataset):
    def __init__(self, hf_dataset, english_tokenizer, japanese_tokenizer, max_len=MAX_LEN):
        self.hf_dataset = hf_dataset
        self.english_tokenizer = english_tokenizer
        self.japanese_tokenizer = japanese_tokenizer
        self.max_len = max_len

        self.src_pad_id = english_tokenizer.token_to_id("[PAD]")
        self.tgt_pad_id = japanese_tokenizer.token_to_id("[PAD]")
        self.tgt_sos_id = japanese_tokenizer.token_to_id("[SOS]")
        self.tgt_eos_id = japanese_tokenizer.token_to_id("[EOS]")

    def __len__(self):
        return len(self.hf_dataset['train'])

    def _pad(self, ids, pad_id):
        ids = ids[:self.max_len]
        ids = ids + [pad_id] * (self.max_len - len(ids))
        return ids

    def __getitem__(self, idx):
        example = self.hf_dataset['train'][idx]
        translation = example['translation']

        src_ids = self.english_tokenizer.encode(translation['en']).ids
        tgt_ids = self.japanese_tokenizer.encode(translation['ja']).ids

        tgt_ids = [self.tgt_sos_id] + tgt_ids + [self.tgt_eos_id]

        src_ids = self._pad(src_ids, self.src_pad_id)
        tgt_ids = self._pad(tgt_ids, self.tgt_pad_id)

        return torch.tensor(src_ids, dtype=torch.long), torch.tensor(tgt_ids, dtype=torch.long)


In [23]:
import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, embedding_dim, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, embedding_dim) #[5000,512]
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1) #[1,5000]
        div_term = torch.exp(torch.arange(0, embedding_dim, 2).float() *
                              (-math.log(10000.0) / embedding_dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))  # [1, max_len, embedding_dim]

    def forward(self, x):
        # x: [batch, seq_len, embedding_dim]
        return x + self.pe[:, :x.size(1), :]

In [24]:
import torch
import torch.nn as nn

class feedforward(nn.Module):
  def __init__(self):
    super().__init__()
    self.layer1 = nn.Sequential(
                  nn.Linear(in_features=512, out_features=256),
                  nn.ReLU(),
                  nn.Dropout(0.1),
                  nn.Linear(in_features=256, out_features=512),
    )

  def forward(self,x):
    return self.layer1(x)


In [25]:
import torch
import torch.nn as nn

class EngToJapDecoder(nn.Module):
    def __init__(self, embedding_dim, num_heads, vocab_size):
        super().__init__()

        self.embedding_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)
        self.positional_encoding = PositionalEncoding(embedding_dim)

        self.masked_att = nn.MultiheadAttention(embedding_dim, num_heads, batch_first=True)
        self.cross_att = nn.MultiheadAttention(embedding_dim, num_heads, batch_first=True)

        self.norm1 = nn.LayerNorm(embedding_dim)
        self.norm2 = nn.LayerNorm(embedding_dim)
        self.norm3 = nn.LayerNorm(embedding_dim)

        self.feed_forward = feedforward()
        self.dropout = nn.Dropout(0.1)

        self.linear = nn.Linear(in_features=embedding_dim, out_features=vocab_size)

    def forward(self, x, encoder_output, src_key_padding_mask, tgt_key_padding_mask, causal_mask):
        embedding = self.embedding_layer(x)
        embedding = self.positional_encoding(embedding)
        embedding = self.dropout(embedding)

        # attention_out, attention_out_weight
        attn_out = self.masked_att(
            embedding, embedding, embedding,
            key_padding_mask=tgt_key_padding_mask,
            attn_mask=causal_mask,
            need_weights=False
        )[0]

        x = self.norm1(embedding + attn_out)

        # attention_out, attention_out_weight
        cross_out = self.cross_att(
            x, encoder_output, encoder_output,
            key_padding_mask=src_key_padding_mask,
            need_weights=False
        )[0]
        x = self.norm2(x + cross_out)

        # feed-forward + residual + norm
        ff_out = self.feed_forward(x)
        x = self.norm3(x + ff_out)

        logits = self.linear(x)
        return logits  # CrossEntropyLoss applies softmax internally


In [26]:
import torch
import torch.nn as nn

class EngToJapEncoder(nn.Module):
    def __init__(self, embedding_dim, num_heads, vocab_size):
        super().__init__()

        self.embedding_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)
        self.positional_encoding = PositionalEncoding(embedding_dim)

        self.self_att = nn.MultiheadAttention(embedding_dim, num_heads, batch_first=True)

        self.norm1 = nn.LayerNorm(embedding_dim)
        self.norm2 = nn.LayerNorm(embedding_dim)

        self.feed_forward = feedforward()
        self.dropout = nn.Dropout(0.1)

    def forward(self, x, src_key_padding_mask):
        embedding = self.embedding_layer(x)
        embedding = self.positional_encoding(embedding)
        embedding = self.dropout(embedding)

        attn_out, _ = self.self_att(
            embedding, embedding, embedding,
            key_padding_mask=src_key_padding_mask
        )
        x = self.norm1(embedding + attn_out)

        ff_out = self.feed_forward(x)
        x = self.norm2(x + ff_out)

        return x


In [27]:
import torch.nn as nn

class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, embedding_dim=512, num_heads=8):
        super().__init__()
        self.encoder = EngToJapEncoder(embedding_dim, num_heads, src_vocab_size)
        self.decoder = EngToJapDecoder(embedding_dim, num_heads, tgt_vocab_size)

    def forward(self, src, tgt, src_key_padding_mask, tgt_key_padding_mask, causal_mask):
        encoder_output = self.encoder(src, src_key_padding_mask)
        decoder_output = self.decoder(tgt, encoder_output, src_key_padding_mask, tgt_key_padding_mask, causal_mask)
        return decoder_output


In [28]:
from torch.utils.data import DataLoader, random_split

full_dataset = TranslationDataset(dataset, english_tokenizer, japanese_tokenizer)

train_size = int(len(full_dataset) * 0.8)
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = Transformer(ENGLISH_VOCAB_SIZE, JAPANESE_VOCAB_SIZE).to(device)
print("Using device:", device)


Using device: cuda


In [29]:
import torch
import torch.optim as optim
import torch.nn as nn

src_PAD = english_tokenizer.token_to_id("[PAD]")
tgt_PAD = japanese_tokenizer.token_to_id("[PAD]")

loss_fn = nn.CrossEntropyLoss(ignore_index=tgt_PAD)

d_model = 512       
warmup_steps = 4000    

optimizer = optim.Adam(
    model.parameters(),
    lr=1.0,
    betas=(0.9, 0.98),
    eps=1e-9
)

def lr_lambda(step):
    step = max(step, 1)
    return (d_model ** -0.5) * min(step ** -0.5, step * (warmup_steps ** -1.5))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

epochs = 50

def make_masks(src, tgt_input, device):
    src_key_padding_mask = (src == src_PAD).to(device)
    tgt_key_padding_mask = (tgt_input == tgt_PAD).to(device)

    seq_len = tgt_input.size(1)
    causal_mask = torch.triu(
        torch.ones(seq_len, seq_len, dtype=torch.bool, device=device), diagonal=1
    )

    return src_key_padding_mask, tgt_key_padding_mask, causal_mask

for epoch in range(1, epochs + 1):
    # Training
    model.train()
    total_train_loss = 0.0

    for src, tgt in train_loader:
        src, tgt = src.to(device), tgt.to(device)

        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]

        src_kpm, tgt_kpm, causal_mask = make_masks(src, tgt_input, device)

        optimizer.zero_grad()

        output = model(src, tgt_input, src_kpm, tgt_kpm, causal_mask)
        loss = loss_fn(output.reshape(-1, output.size(-1)), tgt_output.reshape(-1))

        loss.backward()
        optimizer.step()
        scheduler.step() 

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)

    # Validation
    model.eval()
    total_val_loss = 0.0

    with torch.no_grad():
        for src, tgt in val_loader:
            src, tgt = src.to(device), tgt.to(device)

            tgt_input = tgt[:, :-1]
            tgt_output = tgt[:, 1:]

            src_kpm, tgt_kpm, causal_mask = make_masks(src, tgt_input, device)

            output = model(src, tgt_input, src_kpm, tgt_kpm, causal_mask)
            loss = loss_fn(output.reshape(-1, output.size(-1)), tgt_output.reshape(-1))

            total_val_loss += loss.item()

    avg_val_loss = total_val_loss / len(val_loader)

    print(f"Epoch {epoch}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")


Epoch 1/50 | Train Loss: 4.6839 | Val Loss: 3.4952
Epoch 2/50 | Train Loss: 3.2845 | Val Loss: 2.9836
Epoch 3/50 | Train Loss: 2.9150 | Val Loss: 2.7608
Epoch 4/50 | Train Loss: 2.7253 | Val Loss: 2.6413
Epoch 5/50 | Train Loss: 2.6066 | Val Loss: 2.5606
Epoch 6/50 | Train Loss: 2.5222 | Val Loss: 2.5058
Epoch 7/50 | Train Loss: 2.4577 | Val Loss: 2.4680
Epoch 8/50 | Train Loss: 2.4063 | Val Loss: 2.4358
Epoch 9/50 | Train Loss: 2.3634 | Val Loss: 2.4082
Epoch 10/50 | Train Loss: 2.3267 | Val Loss: 2.3874
Epoch 11/50 | Train Loss: 2.2948 | Val Loss: 2.3705
Epoch 12/50 | Train Loss: 2.2665 | Val Loss: 2.3527
Epoch 13/50 | Train Loss: 2.2419 | Val Loss: 2.3423
Epoch 14/50 | Train Loss: 2.2194 | Val Loss: 2.3285
Epoch 15/50 | Train Loss: 2.1989 | Val Loss: 2.3182
Epoch 16/50 | Train Loss: 2.1805 | Val Loss: 2.3103
Epoch 17/50 | Train Loss: 2.1636 | Val Loss: 2.3032
Epoch 18/50 | Train Loss: 2.1477 | Val Loss: 2.2935
Epoch 19/50 | Train Loss: 2.1328 | Val Loss: 2.2844
Epoch 20/50 | Train L

In [30]:
@torch.no_grad()
def translate(sentence, model, english_tokenizer, japanese_tokenizer, max_len=64, device=device):
    model.eval()

    src_pad_id = english_tokenizer.token_to_id("[PAD]")
    tgt_pad_id = japanese_tokenizer.token_to_id("[PAD]")
    sos_id = japanese_tokenizer.token_to_id("[SOS]")
    eos_id = japanese_tokenizer.token_to_id("[EOS]")

    # encode + pad the source sentence
    src_ids = english_tokenizer.encode(sentence).ids
    src_ids = src_ids[:max_len] + [src_pad_id] * max(0, max_len - len(src_ids))
    src = torch.tensor([src_ids], dtype=torch.long, device=device)
    src_key_padding_mask = (src == src_pad_id)

    # run the encoder once - its output is reused at every decoding step
    encoder_output = model.encoder(src, src_key_padding_mask)

    tgt_ids = [sos_id]

    for _ in range(max_len):
        tgt = torch.tensor([tgt_ids], dtype=torch.long, device=device)
        tgt_key_padding_mask = (tgt == tgt_pad_id)

        seq_len = tgt.size(1)
        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len, dtype=torch.bool, device=device), diagonal=1
        )

        logits = model.decoder(tgt, encoder_output, src_key_padding_mask, tgt_key_padding_mask, causal_mask)
        next_id = logits[0, -1].argmax(dim=-1).item()

        tgt_ids.append(next_id)

        if next_id == eos_id:
            break

    output_ids = tgt_ids[1:]
    if output_ids and output_ids[-1] == eos_id:
        output_ids = output_ids[:-1]

    return japanese_tokenizer.decode(output_ids)


output = translate("Hello, how are you?", model, english_tokenizer, japanese_tokenizer)
print(output)

「 そ ##の ##は 、 い ##つ ##は ##ど ##う ##か 。 そ ##の ##か ##は ##ど ##う ##か 。 そ ##の ##あ ##る ##か ##は ##ど ##う ##か 。 そ ##の ##か 。
